#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [ ]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)
